# TimeSCape v0.2 — Python Demo Notebook

**Circadian rhythm detection in single-cell RNA-seq data**  
Selim Romero · Texas A&M University · ssromerogon@tamu.edu

---

This notebook mirrors the workflow from the original `circadian_analysis.ipynb` but uses the
formal `timescape` Python package.  All analysis functions are now importable, documented,
and statistically equivalent to the MATLAB v0.2 and R (TimeSCape_R) implementations.

**Pipeline overview:**
1. Load AnnData (h5ad)
2. Build ZT metadata table (`tmeta`)
3. Run `run_timescape()` → 6 CSV outputs per cell type
4. Generate heatmaps and gene plots
5. (Optional) Loop over tumor stages

**Key fix vs. original notebook:**  
The original notebook used `time_step × arange(n_zts)` as time values, assuming evenly-spaced
ZT points.  `timescape` always passes the **actual numeric ZT hours** to the cosine fitter,
so missing time points (e.g. a rare cell type absent at ZT12) are handled correctly without
imputation.

## 0. Installation

Run once, then restart the kernel.

```bash
# Option A — conda environment (recommended)
conda env create -f environment.yml
conda activate timescape

# Option B — pip into an existing environment
pip install -e .   # from the TimeSCape_py directory
```

In [ ]:
# --- 0. IMPORTS ---
import os
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import hdf5plugin

# TimeSCape package
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))  # add parent if needed

from timescape import (
    run_timescape,
    build_tmeta,
    plot_gene_single,
    generate_heatmap,
    save_batch_plots,
)

import timescape
print(f"TimeSCape version: {timescape.__version__}")
sc.settings.verbosity = 1

## 1. Load Data

In [ ]:
# --- 1. LOAD DATA ---
# Update this path to your h5ad file.
DATA_PATH = r"Z:\selim_working_dir\2025_sato_anestacia_circadian_rhythm\scanpy_analysis\sato_mouse_breast_cancer_circadian.h5ad"

print(f"Loading: {DATA_PATH}")
adata = sc.read_h5ad(DATA_PATH)
print(adata)
print("\nObs columns:", list(adata.obs.columns))

In [ ]:
# --- 1b. INSPECT METADATA ---
# Identify the correct ZT column and cell type column.

ZT_COL       = 'ZT_time'          # column holding ZT labels like 'ZT06'
CELLTYPE_COL = 'sub_cell_types'   # or 'broad_cell_types', 'cell_type'

print(f"Unique ZT values in '{ZT_COL}':")
print(sorted(adata.obs[ZT_COL].dropna().unique()))

print(f"\nUnique cell types in '{CELLTYPE_COL}':")
print(sorted(adata.obs[CELLTYPE_COL].dropna().unique()))

## 2. Build ZT Metadata (`tmeta`)

`build_tmeta()` auto-parses ZT labels like `'ZT06'`, `'ZT12'`, etc.  
If your labels use a different format, build the table manually (see commented example below).

In [ ]:
# --- 2. BUILD TMETA ---
unique_zt = sorted(adata.obs[ZT_COL].dropna().unique())

# Auto-parse from ZT string labels (ZT00, ZT03, ZT06, ..., ZT21)
tmeta = build_tmeta(unique_zt)
print(tmeta)

# --- MANUAL TMETA (use this if auto-parse fails) ---
# tmeta = pd.DataFrame({
#     'old_labels': ['ZT00', 'ZT03', 'ZT06', 'ZT09', 'ZT12', 'ZT15', 'ZT18', 'ZT21'],
#     'new_labels': ['ZT00', 'ZT03', 'ZT06', 'ZT09', 'ZT12', 'ZT15', 'ZT18', 'ZT21'],
#     'ZT_times':   [0,       3,       6,       9,       12,      15,      18,      21],
# })

## 3. Run TimeSCape

### 3a. Quick test — one cell type

In [ ]:
# --- 3a. SINGLE CELL TYPE RUN ---
OUT_DIR = "./timescape_output"

T1, T2 = run_timescape(
    adata         = adata,
    tmeta         = tmeta,
    celltype_col  = CELLTYPE_COL,
    zt_col        = ZT_COL,
    custom_celltype = ['CD8+ T cells'],  # restrict to one cell type for speed
    period        = 24,
    norm_str      = 'lib_size',          # normalize from raw counts
    rm_low_conf   = True,
    plot_heat     = True,
    outdir        = OUT_DIR,
    n_jobs        = -1,                  # use all CPUs
)

print(f"\nTop confident genes:")
T1.head(10)

### 3b. Full run — all cell types

In [ ]:
# --- 3b. FULL RUN (all cell types) ---
# This may take several minutes for large datasets.

T1_all, T2_all = run_timescape(
    adata         = adata,
    tmeta         = tmeta,
    celltype_col  = CELLTYPE_COL,
    zt_col        = ZT_COL,
    period        = 24,
    norm_str      = 'lib_size',
    rm_low_conf   = True,
    plot_heat     = True,
    outdir        = OUT_DIR,
    n_jobs        = -1,
)

# Summary is written to OUT_DIR/all_cell_types_period_24_summary_results.csv
summary = pd.read_csv(os.path.join(OUT_DIR, 'all_cell_types_period_24_summary_results.csv'))
summary

## 4. Visualizations

In [ ]:
# --- 4a. HEATMAP ---
# Reads the confident CSV files written by run_timescape().

generate_heatmap(
    celltype  = 'CD8+ T cells',
    outdir    = os.path.join(OUT_DIR, 'CD8_T_cells'),
    period    = 24,
    strict    = True,    # BH-adjusted p < 0.05
    circ_only = False,   # True = clock genes only
)

In [ ]:
# --- 4b. SINGLE GENE PLOT ---
# Blue line = cosine fit | Orange = per-ZT means | Red dashed = acrophase

fig = plot_gene_single(
    adata        = adata,
    gene         = 'Per2',
    celltype     = 'CD8+ T cells',
    tmeta        = tmeta,
    celltype_col = CELLTYPE_COL,
    zt_col       = ZT_COL,
    period       = 24,
    norm_str     = 'lib_size',
    show_cells   = True,
    use_violin   = False,
)
fig.savefig('Per2_CD8_Tcells.png', dpi=150, bbox_inches='tight')

In [ ]:
# --- 4c. BATCH GENE PLOTS ---
# Saves one PNG per gene to OUT_DIR/<CellType>/plots_confident/

save_batch_plots(
    adata        = adata,
    tmeta        = tmeta,
    celltype     = 'CD8+ T cells',
    outdir       = OUT_DIR,
    celltype_col = CELLTYPE_COL,
    zt_col       = ZT_COL,
    period       = 24,
    plot_type    = 1,    # 1=confident | 2=non-confident | 3=clock genes
    max_genes    = 50,
    show_cells   = True,
    use_violin   = False,
)

## 5. Loop Over Tumor Stages

Reproduces the stage-stratified analysis from the original notebook.

In [ ]:
# --- 5. STAGE-STRATIFIED ANALYSIS ---
STAGE_COL  = 'tumor_stage'         # column with Early / Intermediate / Advanced
STAGE_DIR  = './timescape_by_stage'
os.makedirs(STAGE_DIR, exist_ok=True)

all_stages = sorted(adata.obs[STAGE_COL].dropna().unique())
print(f"Stages found: {all_stages}")

for stage in all_stages:
    print(f"\n{'='*60}")
    print(f"STAGE: {stage}")
    print(f"{'='*60}")

    stage_safe = stage.replace(' ', '_').replace('/', '_')
    stage_outdir = os.path.join(STAGE_DIR, f'stage_{stage_safe}')

    idx = adata.obs[STAGE_COL] == stage
    adata_stage = adata[idx, :].copy()
    print(f"  Cells: {adata_stage.n_obs:,}")

    try:
        run_timescape(
            adata         = adata_stage,
            tmeta         = tmeta,
            celltype_col  = CELLTYPE_COL,
            zt_col        = ZT_COL,
            period        = 24,
            norm_str      = 'lib_size',
            rm_low_conf   = True,
            plot_heat     = True,
            outdir        = stage_outdir,
            n_jobs        = -1,
        )
        print(f"  Stage {stage} complete → {stage_outdir}")
    except Exception as err:
        print(f"  ERROR in stage {stage}: {err}")

print("\nAll stages processed.")

## 6. Core API Reference

```python
from timescape import (
    run_timescape,      # main pipeline
    build_tmeta,        # auto-build ZT metadata table
    estimate_phase_r,   # fit one gene (cosinor + F-test + Pearson)
    generate_heatmap,   # Z-score heatmap from CSV output
    plot_gene_single,   # cosine fit plot for one gene
    save_batch_plots,   # export PNG per gene
    bh_adjust,          # Benjamini-Hochberg FDR correction
    wrap_acrophase,     # wrap acrophase into [0, period)
)
```

### `run_timescape()` parameter summary

| Parameter | Default | Description |
|-----------|---------|-------------|
| `adata` | — | AnnData object |
| `tmeta` | — | ZT metadata DataFrame (old_labels, new_labels, ZT_times) |
| `celltype_col` | `'cell_type'` | obs column for cell type |
| `zt_col` | `'ZT_time'` | obs column for ZT labels |
| `period` | `24.0` | Circadian period (24 or 12) |
| `norm_str` | `'lib_size'` | `'lib_size'` / `'logcounts'` / `'none'` |
| `rm_low_conf` | `True` | Write confident-only output CSVs |
| `plot_heat` | `True` | Generate heatmap PNG per cell type |
| `test_type` | `'Ftest'` | `'Ftest'` or `'LRT'` |
| `custom_celltype` | `None` | Restrict to these cell types |
| `custom_genelist` | `None` | Restrict to these genes |
| `n_jobs` | `-1` | joblib parallel workers |
| `outdir` | `'.'` | Root output directory |

### Statistical model

For each gene g and cell type, `TimeSCape` fits:

```
f(t) = A · cos( 2π(t − φ) / T ) + M
```

to all individual cell expression values (not means), then applies:

1. **F-test** on single-cell residuals: `F = [(SSR_null − SSR_cosine)/2] / [SSR_cosine/(N−3)]`
2. **Pearson correlation** between per-ZT-point mean expression R0 and the fitted cosine

A gene is **confident** if both p-values < 0.05.  
Both sets of p-values are also BH-adjusted for multiple testing.